**Training of Computer Vision Model**

In [ ]:
!pip install ultralytics

In [ ]:
from pathlib import Path

root = Path("../../data/raw_data/final_unified_dataset")
print("Dataset exists:", root.exists())
print("Train images:", (root / "images/train").exists())
print("Val images:", (root / "images/val").exists())
print("Test images:", (root / "images/test").exists())
print("Train labels:", (root / "labels/train").exists())
print("Val labels:", (root / "labels/val").exists())
print("Test labels:", (root / "labels/test").exists())
print("YAML exists:", (root / "dataset.yaml").exists())
print(Path("../../data/raw_data/final_unified_dataset/dataset.yaml").read_text())

In [ ]:
from ultralytics import YOLO
model = YOLO("yolov8n.pt")
results = model.train(
    data="../../data/raw_data/final_unified_dataset/dataset.yaml",
    epochs=20,
    imgsz=640,
    batch=8,
    device = "mps"
)
metrics = model.val()
print(metrics)

In [ ]:
model = YOLO("yolov8s.pt") 

results = model.train(
    data="../../data/raw_data/final_unified_dataset/dataset.yaml",
    epochs=20,
    imgsz=640,
    batch=8,
    device="mps",
    workers=4,
    patience=20,
    project="yolov8s"
)

In [ ]:
model = YOLO("yolov8n.pt") 

results = model.train(
    data="../../data/raw_data/final_unified_dataset/dataset.yaml",
    epochs=20,
    imgsz=640,
    batch=10,
    device="mps",
    workers=2,
    patience=20,
    project="yolov8n"
)


In [ ]:
import os
from collections import Counter

label_dir = "../../data/raw_data/final_unified_dataset/labels/train"
class_counts = Counter()

for fname in os.listdir(label_dir):
    if not fname.endswith(".txt"):
        continue
    with open(os.path.join(label_dir, fname)) as f:
        for line in f:
            parts = line.strip().split()
            if parts:
                class_counts[int(parts[0])] += 1

print("Class distribution in val labels:")
for cls_id, count in sorted(class_counts.items()):
    print(f"  Class {cls_id}: {count} instances")

In [ ]:
from ultralytics import YOLO

model = YOLO("yolo11n.pt")

results = model.train(
    data="../../data/raw_data/final_unified_dataset/dataset.yaml",
    epochs=20,              
    imgsz=640,
    batch=10,
    device="mps",
    workers=0,
    patience=20,
    project="yolo11n",
)

In [ ]:
import torch

print("MPS available:", torch.backends.mps.is_available())
print("MPS built:", torch.backends.mps.is_built())

In [ ]:
import torch
import platform

print(f"OS:               {platform.system()}")
print(f"PyTorch version:  {torch.__version__}")
print(f"MPS available:    {torch.backends.mps.is_available()}")
print(f"CPU count:        {torch.get_num_threads()}")

# Test if multiprocessing actually works
import torch.multiprocessing as mp
try:
    mp.set_start_method("spawn", force=True)
    print("Multiprocessing: spawn method set successfully")
except RuntimeError as e:
    print(f"Multiprocessing warning: {e}")

In [ ]:
from torch.utils.data import DataLoader, TensorDataset
import torch
import time

# Create a dummy dataset
dummy = TensorDataset(torch.randn(100, 3, 640, 640))

# Test with workers=0
loader0 = DataLoader(dummy, batch_size=10, num_workers=0)
start = time.time()
for batch in loader0:
    pass
time0 = time.time() - start

# Test with workers=2
loader2 = DataLoader(dummy, batch_size=10, num_workers=2)
start = time.time()
for batch in loader2:
    pass
time2 = time.time() - start

print(f"workers=0: {time0:.2f}s")
print(f"workers=2: {time2:.2f}s")
print(f"Speedup:   {time0/time2:.2f}x")

if time2 < time0:
    print("Workers ARE helping")
else:
    print("Workers are NOT helping — MPS multiprocessing issue")